# Convert the 400-episode FR5 set → LeRobot v2, push to the Hub

For the 2026-07 **multi-task** raw set (`Slifold/episodes_20260717`, 400 episodes,
9 canonical tasks, 400 unique language instructions). Produces
`<you>/fr5-pick-place-lerobot-v2` and leaves the v1 repo untouched so old runs
stay reproducible.

**Architecture note:** the Hub copy ships **videos only** — extracted frames for
400 episodes are ~1.1M JPEGs / ~50 GB, which is the wrong thing to put in a Hub
repo. The training notebooks extract frames locally from the videos on first run
(one-time, ~10–30 min on the pod). That also means this notebook needs no GPU
and only modest disk (~15–25 GB) — a cheap CPU pod is fine.

Run top to bottom. Every step verifies before the next runs.

## 1 · Parameters

In [ ]:
import os

HF_TOKEN            = os.environ.get("HF_TOKEN", "")     # read source + write target
SOURCE_RAW_REPO     = "Slifold/episodes_20260717"        # 400 raw episodes (input)
TARGET_LEROBOT_REPO = "<you>/fr5-pick-place-lerobot-v2"  # output (created private)

CAMERAS = "wrist,scene"     # both views — every policy trains on 2 cameras

# DO NOT set a task override. The whole value of this dataset is its 400 unique
# per-episode instructions over 9 canonical tasks; an override collapses them to
# one string and destroys the language signal. Smoke tests only.
TASK_OVERRIDE = ""

EXPECTED_EPISODES = 400     # hard-checked before converting and before pushing

RAW_DIR = "/workspace/raw_episodes_v2"
OUT_DIR = "/workspace/lerobot_dataset_v2"

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "need an HF token (read source + write target)"
assert TARGET_LEROBOT_REPO != "<you>/fr5-pick-place-lerobot-v2", "set TARGET_LEROBOT_REPO"
print("parameters set")

## 2 · Install dependencies

In [ ]:
import subprocess, sys, importlib

# Deliberately NOT quiet — pip's resolver messages are the only warning you get
# before a wrong version bites mid-conversion.
subprocess.run([sys.executable, "-m", "pip", "install",
                "huggingface_hub>=0.34", "opencv-python-headless>=4.9",
                "pandas", "pyarrow", "numpy", "tqdm"], check=True)

_missing = []
for _m in ("huggingface_hub", "cv2", "pandas", "pyarrow", "numpy", "tqdm"):
    try:
        importlib.import_module(_m)
    except Exception as _e:
        _missing.append(f"{_m}: {type(_e).__name__}: {_e}")
if _missing:
    raise SystemExit("dependency check FAILED:\n  " + "\n  ".join(_missing))
print("all imports OK")

## 3 · HuggingFace login + access check

In [ ]:
from huggingface_hub import login, whoami, HfApi

login(token=HF_TOKEN)
print("logged in as:", whoami()["name"])
# fail here, not after a 15 GB download
n_files = len(HfApi().list_repo_files(SOURCE_RAW_REPO, repo_type="dataset"))
print(f"source repo reachable: {SOURCE_RAW_REPO} ({n_files} files)")

## 4 · Download the raw episodes (~15 GB)

In [ ]:
import shutil
from huggingface_hub import snapshot_download

free_gb = shutil.disk_usage("/workspace").free / 1e9
print(f"free disk on /workspace: {free_gb:.0f} GB")
assert free_gb > 40, "need ~40 GB free (raw download + converted output)"

snapshot_download(SOURCE_RAW_REPO, repo_type="dataset", local_dir=RAW_DIR)
print("downloaded ->", RAW_DIR)

## 5 · Pre-flight: verify the raw set BEFORE converting

Counts and completeness, per-episode language present everywhere, camera rate,
and the canonical-task balance table. Aborts loudly rather than converting bad
data.

In [ ]:
import json, pathlib, collections, statistics
import numpy as np

raw = pathlib.Path(RAW_DIR)
eps = sorted(raw.glob("episode_*"))
assert len(eps) == EXPECTED_EPISODES, f"expected {EXPECTED_EPISODES} episodes, found {len(eps)}"

REQ = ["data.csv", "wrist_cam.mp4", "scene_cam.mp4", "meta.json",
       "wrist_cam_ts.npy", "scene_cam_ts.npy"]
bad = {e.name: [f for f in REQ if not (e / f).exists()] for e in eps}
bad = {k: v for k, v in bad.items() if v}
assert not bad, f"incomplete episodes: {bad}"

metas = {e.name: json.loads((e / "meta.json").read_text()) for e in eps}
no_instr = [k for k, m in metas.items() if not m.get("instruction")]
no_canon = [k for k, m in metas.items() if not m.get("instruction_canonical")]
assert not no_instr, f"episodes missing instruction: {no_instr[:5]}"
assert not no_canon, f"episodes missing instruction_canonical: {no_canon[:5]}"

instrs = [m["instruction"] for m in metas.values()]
canons = [m["instruction_canonical"] for m in metas.values()]
print(f"{len(eps)} episodes, all complete")
print(f"unique instructions: {len(set(instrs))}   canonical tasks: {len(set(canons))}")
print(f"total demo time: {sum(m['duration_s'] for m in metas.values())/60:.0f} min")

# camera rate spot-check — training and deployment assume 30 fps
for e in (eps[0], eps[len(eps)//2], eps[-1]):
    ts = np.load(e / "wrist_cam_ts.npy")
    fps = 1 / np.median(np.diff(ts))
    assert abs(fps - 30.0) < 0.5, f"{e.name}: wrist cam at {fps:.1f} fps, expected 30"
print("camera rate: 30 fps confirmed on first/middle/last episodes")

print("\ncanonical task balance:")
for task, n in collections.Counter(canons).most_common():
    print(f"   {n:3d}x  {task}")

## 6 · Fetch the tested converter (pinned @ `9cd91a6791fa`)

In [ ]:
import urllib.request

RAW_URL = ("https://raw.githubusercontent.com/SreevaatsavB/fairino-fr5-policies/"
           "9cd91a6791fa144026a4421572353eb1bbf32b63/common/convert_episodes.py")
urllib.request.urlretrieve(RAW_URL, "convert_episodes.py")
print("fetched convert_episodes.py (pinned @ 9cd91a6791fa)")

## 7 · Convert raw → LeRobot

Timestamp-based resampling onto the 30 fps camera clock (the raw CSV rate varies
33–65 Hz and that is fine — no fixed-rate assumption anywhere). No
`--extract-frames`: the Hub copy ships videos only; training pods extract
locally. No `--task`: per-episode instructions flow through.

In [ ]:
import subprocess, sys

cmd = [sys.executable, "convert_episodes.py",
       "--episodes", RAW_DIR, "--out", OUT_DIR, "--cameras", CAMERAS]
if TASK_OVERRIDE:      # smoke tests only — destroys the language signal
    print("WARNING: TASK_OVERRIDE set — collapsing 400 instructions to one string!")
    cmd += ["--task", TASK_OVERRIDE]
subprocess.run(cmd, check=True)

## 8 · Write `meta/canonical_tasks.json`

`tasks.parquet` carries the 400 varied instructions (that is what trains). This
extra file maps episode → canonical task so evals can group MAE/success by the
9 canonical tasks without re-touching the raw repo.

In [ ]:
import json, pathlib

canonical = {}
for e in sorted(pathlib.Path(RAW_DIR).glob("episode_*")):
    m = json.loads((e / "meta.json").read_text())
    idx = int(e.name.split("_")[1])
    canonical[idx] = {"canonical": m["instruction_canonical"],
                      "instruction": m["instruction"]}
out = pathlib.Path(OUT_DIR, "meta", "canonical_tasks.json")
out.write_text(json.dumps(canonical, indent=1))
print(f"wrote {out} ({len(canonical)} episodes, "
      f"{len(set(v['canonical'] for v in canonical.values()))} canonical tasks)")

## 9 · Verify the converted dataset

In [ ]:
import json, pathlib
import pyarrow.parquet as pq

info = json.loads(pathlib.Path(OUT_DIR, "meta", "info.json").read_text())
sd = info["features"]["observation.state"]["shape"][0]
ad = info["features"]["action"]["shape"][0]
cams = [k for k in info["features"] if k.startswith("observation.images.")]
print(f"episodes = {info['total_episodes']}   frames = {info['total_frames']}   fps = {info['fps']}")
print(f"state_dim = {sd}   action_dim = {ad}   cameras = {cams}")
assert info["total_episodes"] == EXPECTED_EPISODES
assert sd == 7 and ad == 7 and info["fps"] == 30
assert {"observation.images.wrist_cam", "observation.images.scene_cam"} <= set(cams)

tasks = pq.read_table(pathlib.Path(OUT_DIR, "meta", "tasks.parquet")).to_pandas()
print(f"tasks: {len(tasks)} rows, {tasks['task'].nunique()} unique")
assert tasks["task"].nunique() == EXPECTED_EPISODES, "expected one unique instruction per episode"
for t in tasks["task"].head(3): print("   e.g.", t)

# every episode has both videos on disk
vids = pathlib.Path(OUT_DIR, "videos")
for cam in cams:
    n = len(list((vids / cam).glob("chunk-*/file-*.mp4")))
    assert n == EXPECTED_EPISODES, f"{cam}: {n} videos"
print("both cameras: 400/400 videos present")

# spot-check one mid-dataset row: state/action look like joint degrees + gripper
df = pq.read_table(pathlib.Path(OUT_DIR, "data/chunk-000/file-000.parquet")).to_pandas()
row = df.iloc[len(df) // 2]
print("mid row  state:", [round(x, 1) for x in row["observation.state"]])
print("mid row action:", [round(x, 1) for x in row["action"]])
print(f"total rows: {len(df)}  (~{len(df)/30/60:.0f} min at 30 Hz)")

## 10 · Push to the Hub (videos + parquet + meta — no frames)

~15 GB. `frames/` is excluded by design; training pods rebuild it locally from
the videos in ~10–30 min (the training notebooks do this automatically).

In [ ]:
import pathlib
from huggingface_hub import HfApi

total = sum(f.stat().st_size for f in pathlib.Path(OUT_DIR).rglob("*") if f.is_file()) / 1e9
print(f"uploading ~{total:.1f} GB from {OUT_DIR}")

api = HfApi()
api.create_repo(TARGET_LEROBOT_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=TARGET_LEROBOT_REPO, repo_type="dataset",
                  ignore_patterns=["frames/**"],
                  commit_message=f"FR5 multi-task v2: {EXPECTED_EPISODES} episodes, "
                                 f"9 canonical tasks, per-episode instructions")
print(f"pushed -> https://huggingface.co/datasets/{TARGET_LEROBOT_REPO}")

## Done — next steps

In `train_pi0_runpod.ipynb` / `train_pi05_runpod.ipynb` set:

```python
HF_DATASET_REPO = "<you>/fr5-pick-place-lerobot-v2"
```

First run on a training pod will print `extracting frames ...` once (~10–30 min),
then train as usual. Recommended first run: the pi0.5 30k-step recipe — this is
the dataset its language context exists for.